In [315]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import numpy as np
from scipy import signal
import matplotlib.colors as colors
from ipywidgets import interact, widgets, interactive_output

# --- CONFIGURATION ---
FOLDER = "stitch_tests"
SAMPLE_RATE = 50 

# --- EXPERIMENT PARAMETERS ---
# Adjust these to change the look of your "blocks"
N_PER_SEG = 256 
N_OVERLAP = 230   
N_FFT = 300    

# --- DISPLAY PARAMETERS ---
# V_MIN: Increase this (e.g., -20 or -10) to black out background noise
V_MIN = -20
COLOR_MAP_ACCEL = 'inferno' 
COLOR_MAP_GYRO = 'hot'

# --- FILTER PARAMETERS ---
LOW_CUT = 0.5    # Removes gravity/slow drift
HIGH_CUT = 10  # Removes high-frequency jitter
FILTER_ORDER = 4

In [316]:
def load_file(file_path):
    # load the CSV and show column names
    df = pd.read_csv(file_path)
    return df

def normalize_time(df):
    """Subtracts the first timestamp so time starts at 0."""
    df_norm = df.copy()
    df_norm['sec'] = (df_norm['ms'] - df_norm['ms'].iloc[0]) / 1000.0
    return df_norm

In [317]:
def add_raw_plot(ax, time, data_dict, ylabel):
    """
    Helper to plot multiple lines on a single axis with consistent styling.
    data_dict: {'label': (column_data, color)}
    """
    for label, (value, color) in data_dict.items():
        ax.plot(time, value, label=label, color=color, alpha=0.7)
    
    ax.set_ylabel(ylabel)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

def plot_raw_6axis(df):
    time_zeroed = (df['ms'] - df['ms'].iloc[0]) / 1000.0
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 3), sharex=True)

    # Accelerometer Data Map
    accel_map = {
        'ax': (df['ax'], 'red'),
        'ay': (df['ay'], 'green'),
        'az': (df['az'], 'blue')
    }
    
    # Gyroscope Data Map
    gyro_map = {
        'gx': (df['gx'], 'orange'),
        'gy': (df['gy'], 'purple'),
        'gz': (df['gz'], 'brown')
    }

    # Use the helper function for both subplots
    add_raw_plot(ax1, time_zeroed, accel_map, "Accel (Raw)")
    add_raw_plot(ax2, time_zeroed, gyro_map, "Gyro (Raw)")

    ax2.set_xlabel("Time (seconds)")
    plt.tight_layout()
    plt.show()

In [318]:
def apply_filter(data, fs, cutoffs, btype='band', order=4):
    """
    data: the raw signal
    cutoffs: a single float (for low/high) or a list [low, high] (for band)
    btype: 'low', 'high', or 'band'
    """
    nyq = 0.5 * fs
    # Normalize the cutoffs
    if isinstance(cutoffs, list):
        Wn = [c / nyq for c in cutoffs]
    else:
        Wn = cutoffs / nyq
        
    b, a = signal.butter(order, Wn, btype=btype)
    return signal.filtfilt(b, a, data)

In [319]:
def compute_spectrogram(data, fs, nperseg, noverlap, nfft):
    # nfft must be >= nperseg, so we'll handle that logic here
    actual_nfft = max(nfft, nperseg)
    f, t, Sxx = signal.spectrogram(data, fs=fs, nperseg=nperseg, 
                                   noverlap=noverlap, nfft=actual_nfft)
    s_log = 10 * np.log10(Sxx + 1e-10)
    return f, t, s_log

In [320]:
def draw_workbench_plots(time, raw, filtered, spec_data, vmin, title):
    f, t, s_log = spec_data
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4), sharex=True)
    
    # --- Top: Time Domain ---
    ax1.plot(time, raw, label='Raw', color='gray', alpha=0.3)
    ax1.plot(time, filtered, label='Filtered', color='crimson', lw=1.2)
    ax1.set_title(title)
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.2)

    # --- Bottom: Frequency Domain ---
    im = ax2.pcolormesh(t, f, s_log, shading='gouraud', cmap='hot', vmin=vmin)
    ax2.set_ylabel("Freq (Hz)")
    ax2.set_ylim(0, 15) 
    ax2.set_xlabel("Time (seconds)")
    
    plt.tight_layout()
    plt.show()

In [321]:
def run_filter_workbench(df):
    time_zeroed = (df['ms'] - df['ms'].iloc[0]) / 1000.0
    fs = 50

    # --- UI: FILTER CONTROLS ---
    ui_column = widgets.Dropdown(options=df.columns.drop(['ms', 'sec'], errors='ignore').tolist(), description='Axis:')
    ui_type = widgets.Dropdown(options=['band', 'low', 'high'], value='band', description='Type:')
    ui_low = widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=0.5, description='High-Pass:')
    ui_high = widgets.FloatSlider(min=5.0, max=24.0, step=1.0, value=10.0, description='Low-Pass:')
    
    # --- UI: SPECTROGRAM CONTROLS ---
    ui_vmin = widgets.IntSlider(min=-100, max=0, step=5, value=-30, description='Vmin (Gate):')
    ui_nperseg = widgets.IntSlider(min=8, max=256, step=8, value=32, description='Window (p):')
    ui_noverlap = widgets.IntSlider(min=4, max=250, step=4, value=30, description='Overlap (o):')
    ui_nfft = widgets.IntSlider(min=32, max=512, step=32, value=64, description='FFT Size:')

    def update(column, filter_type, low_f, high_f, vmin, nperseg, noverlap, nfft):
        # 1. Filter
        raw_val = df[column]
        cutoffs = [low_f, high_f] if filter_type == 'band' else (low_f if filter_type == 'high' else high_f)
        processed = apply_filter(raw_val, fs, cutoffs, filter_type)
        
        # 2. Spectrogram (with new variables)
        # We ensure overlap is always less than nperseg to avoid errors
        safe_overlap = min(noverlap, nperseg - 1)
        spec_data = compute_spectrogram(processed, fs, nperseg, safe_overlap, nfft)
        
        # 3. Plot (using the draw function from before)
        draw_workbench_plots(time_zeroed, raw_val, processed, spec_data, vmin, f"Refining Axis: {column}")

    # Layout: Stacking all controls in a long vertical box
    controls = widgets.VBox([
        widgets.Label("Signal Filtering"), ui_column, ui_type, ui_low, ui_high,
        widgets.Label("Spectrogram Resolution"), ui_vmin, ui_nperseg, ui_noverlap, ui_nfft
    ])
    
    out = interactive_output(update, {
        'column': ui_column, 'filter_type': ui_type, 'low_f': ui_low, 'high_f': ui_high,
        'vmin': ui_vmin, 'nperseg': ui_nperseg, 'noverlap': ui_noverlap, 'nfft': ui_nfft
    })
    
    display(widgets.HBox([controls, out]))

In [322]:
def compute_magnitude(df, axes, fs, cutoffs, f_type, order, mode='Pre-Filter'):
    if mode == 'Pre-Filter':
        # Filter each axis individually first
        f_x = apply_filter(df[axes[0]], fs, cutoffs, f_type, order)
        f_y = apply_filter(df[axes[1]], fs, cutoffs, f_type, order)
        f_z = apply_filter(df[axes[2]], fs, cutoffs, f_type, order)
        mag = np.sqrt(f_x**2 + f_y**2 + f_z**2)
    else:
        # Calculate raw magnitude first, then filter
        raw_mag = np.sqrt(df[axes[0]]**2 + df[axes[1]]**2 + df[axes[2]]**2)
        mag = apply_filter(raw_mag, fs, cutoffs, f_type, order)
    
    # Normalize by max as we planned
    mag /= np.max(np.abs(mag))
    return mag

In [323]:
def draw_full_magnitude_workbench(time, acc_mag, gyr_mag, acc_spec, gyr_spec, vmin, title):
    # 4 Subplots: Accel Time, Accel Spec, Gyro Time, Gyro Spec
    fig, (ax_at, ax_as, ax_gt, ax_gs) = plt.subplots(4, 1, figsize=(4, 3), sharex=True)
    
    # --- ROW 1: ACCEL MAG (Time) ---
    ax_at.plot(time, acc_mag, color='crimson', lw=1.2)
    ax_at.set_title(f"{title} | ACCEL MAG")
    ax_at.set_ylabel("Normalized Mag")
    ax_at.grid(True, alpha=0.2)

    # --- ROW 2: ACCEL MAG (Spectrogram) ---
    f_a, t_a, s_a = acc_spec
    ax_as.pcolormesh(t_a, f_a, s_a, shading='gouraud', cmap='inferno', vmin=vmin)
    ax_as.set_ylabel("Freq (Hz)")
    ax_as.set_ylim(0, 15)

    # --- ROW 3: GYRO MAG (Time) ---
    ax_gt.plot(time, gyr_mag, color='orange', lw=1.2)
    ax_gt.set_title("GYRO MAG")
    ax_gt.set_ylabel("Normalized Mag")
    ax_gt.grid(True, alpha=0.2)

    # --- ROW 4: GYRO MAG (Spectrogram) ---
    f_g, t_g, s_g = gyr_spec
    ax_gs.pcolormesh(t_g, f_g, s_g, shading='gouraud', cmap='hot', vmin=vmin)
    ax_gs.set_ylabel("Freq (Hz)")
    ax_gs.set_ylim(0, 15)
    ax_gs.set_xlabel("Time (seconds)")
    
    plt.tight_layout()
    plt.show()

In [324]:
def run_magnitude_workbench(df):
    dt = df['ms'].diff().mean() / 1000.0 
    actual_fs = 1.0 / dt
    
    time_zeroed = (df['ms'] - df['ms'].iloc[0]) / 1000.0

    # 1. UI: Only the controls that matter for Magnitude
    ui_mode = widgets.Dropdown(options=['Pre-Filter', 'Post-Filter'], value='Pre-Filter', description='Order:')
    ui_type = widgets.Dropdown(options=['band', 'low', 'high'], value='band', description='Filt Type:')
    ui_low = widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=0.5, description='High-Pass:')
    ui_high = widgets.FloatSlider(min=5.0, max=24.0, step=1.0, value=10.0, description='Low-Pass:')
    ui_vmin = widgets.IntSlider(min=-100, max=0, step=5, value=-30, description='Vmin (Gate):')
    ui_p = widgets.IntSlider(min=8, max=128, step=8, value=32, description='Window (p):')
    ui_o = widgets.IntSlider(min=4, max=120, step=4, value=30, description='Overlap (o):')

    def update(mode, filter_type, low_f, high_f, vmin, nperseg, noverlap):
        cutoffs = [low_f, high_f] if filter_type == 'band' else (low_f if filter_type == 'high' else high_f)
        safe_o = min(noverlap, nperseg-1)
        
        # Calculate Magnitudes using the sliders
        acc_mag = compute_magnitude(df, ['ax', 'ay', 'az'], actual_fs, cutoffs, filter_type, 4, mode)
        gyr_mag = compute_magnitude(df, ['gx', 'gy', 'gz'], actual_fs, cutoffs, filter_type, 4, mode)
        
        # Calculate Spectrograms
        acc_spec = compute_spectrogram(acc_mag, actual_fs, nperseg, safe_o, 64)
        gyr_spec = compute_spectrogram(gyr_mag, actual_fs, nperseg, safe_o, 64)

        # Plot 4 rows: Accel Mag (Time), Accel Mag (Spec), Gyro Mag (Time), Gyro Mag (Spec)
        fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 14), sharex=True)
        
        # Accel
        ax1.plot(time_zeroed, acc_mag, color='crimson', lw=1.2)
        ax1.set_title(f"ACCEL MAGNITUDE (Mode: {mode})")
        ax2.pcolormesh(acc_spec[1], acc_spec[0], acc_spec[2], shading='gouraud', cmap='inferno', vmin=vmin)
        ax2.set_ylim(0, 15)
        
        # Gyro
        ax3.plot(time_zeroed, gyr_mag, color='orange', lw=1.2)
        ax3.set_title("GYRO MAGNITUDE")
        ax4.pcolormesh(gyr_spec[1], gyr_spec[0], gyr_spec[2], shading='gouraud', cmap='hot', vmin=vmin)
        ax4.set_ylim(0, 15)
        ax4.set_xlabel("Time (seconds)")
        
        plt.tight_layout()
        plt.show()

    controls = widgets.VBox([ui_mode, ui_type, ui_low, ui_high, ui_vmin, ui_p, ui_o])
    out = interactive_output(update, {
        'mode': ui_mode, 'filter_type': ui_type, 'low_f': ui_low, 
        'high_f': ui_high, 'vmin': ui_vmin, 'nperseg': ui_p, 'noverlap': ui_o
    })
    display(widgets.HBox([controls, out]))

In [326]:
if __name__ == "__main__":
    file_path = "stitch_tests/dc_x2_20260415_140201.csv"
    if os.path.exists(file_path):
        df_loaded = load_file(file_path)
        # plot_raw_6axis(df_loaded)
        #run_filter_workbench(df_loaded)
        run_magnitude_workbench(df_loaded)
    else:
        print(f"❌ File not found: {file_path}")